# European Soccer Data - Descriptive Statistics Analysis

This notebook performs comprehensive descriptive statistics analysis for European soccer leagues:
- **Premier League** (England)
- **Serie A** (Italy) 
- **Bundesliga** (Germany)
- **La Liga** (Spain)

The analysis generates individual league statistics and comparative analysis, outputting results to `src/descriptive_statistics.md`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

## Data Loading and Preparation

In [2]:
# Define data paths
data_dir = Path('../csv/processed')
output_dir = Path('../../../src')

# League configurations
leagues = {
    'Premier League': {
        'file': 'premier_league.csv',
        'country': 'England',
        'color': '#1f77b4'
    },
    'Serie A': {
        'file': 'serie_a.csv',
        'country': 'Italy',
        'color': '#ff7f0e'
    },
    'Bundesliga': {
        'file': 'bundesliga.csv',
        'country': 'Germany',
        'color': '#2ca02c'
    },
    'La Liga': {
        'file': 'la_liga.csv',
        'country': 'Spain',
        'color': '#d62728'
    }
}

# Load all league data
league_data = {}
for league_name, config in leagues.items():
    file_path = data_dir / config['file']
    if file_path.exists():
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df['total_goals'] = df['hometeamgoals'] + df['awayteamgoals']
        league_data[league_name] = df
        print(f"Loaded {league_name}: {len(df):,} matches ({df['season'].min()}-{df['season'].max()})")
    else:
        print(f"Warning: {file_path} not found")

print(f"\nTotal matches across all leagues: {sum(len(df) for df in league_data.values()):,}")

Loaded Premier League: 9,410 matches (2000-2024)
Loaded Serie A: 9,012 matches (2000-2024)
Loaded Bundesliga: 7,522 matches (2000-2024)
Loaded La Liga: 9,008 matches (2000-2024)

Total matches across all leagues: 34,952


## Analysis Functions

In [3]:
def analyze_league_results(df, league_name):
    """Analyze match results for a single league."""
    total_matches = len(df)
    
    # Result counts - using numeric values (1=home win, 0=draw, -1=away win)
    home_wins = len(df[df['hometeamresult'] == 1])
    draws = len(df[df['hometeamresult'] == 0])
    away_wins = len(df[df['hometeamresult'] == -1])
    
    # Percentages
    home_win_pct = (home_wins / total_matches) * 100
    draw_pct = (draws / total_matches) * 100
    away_win_pct = (away_wins / total_matches) * 100
    
    return {
        'league': league_name,
        'total_matches': total_matches,
        'home_wins': home_wins,
        'draws': draws,
        'away_wins': away_wins,
        'home_win_pct': home_win_pct,
        'draw_pct': draw_pct,
        'away_win_pct': away_win_pct,
        'season_range': f"{df['season'].min()}-{df['season'].max()}"
    }

def analyze_league_goals(df, league_name):
    """Analyze goal statistics for a single league."""
    return {
        'league': league_name,
        'avg_home_goals': df['hometeamgoals'].mean(),
        'std_home_goals': df['hometeamgoals'].std(),
        'avg_away_goals': df['awayteamgoals'].mean(),
        'std_away_goals': df['awayteamgoals'].std(),
        'avg_total_goals': df['total_goals'].mean(),
        'std_total_goals': df['total_goals'].std(),
        'min_total_goals': df['total_goals'].min(),
        'max_total_goals': df['total_goals'].max()
    }

def analyze_league_odds(df, league_name):
    """Analyze betting odds for a single league."""
    # Filter out missing odds
    odds_df = df.dropna(subset=['OddHome', 'OddDraw', 'OddAway'])
    
    if len(odds_df) == 0:
        return {
            'league': league_name,
            'matches_with_odds': 0,
            'odds_coverage_pct': 0.0
        }
    
    return {
        'league': league_name,
        'matches_with_odds': len(odds_df),
        'odds_coverage_pct': (len(odds_df) / len(df)) * 100,
        'avg_home_odds': odds_df['OddHome'].mean(),
        'std_home_odds': odds_df['OddHome'].std(),
        'avg_draw_odds': odds_df['OddDraw'].mean(),
        'std_draw_odds': odds_df['OddDraw'].std(),
        'avg_away_odds': odds_df['OddAway'].mean(),
        'std_away_odds': odds_df['OddAway'].std()
    }

## Individual League Analysis

In [4]:
# Analyze each league individually
league_results = []
league_goals = []
league_odds = []

for league_name, df in league_data.items():
    print(f"\n=== {league_name} Analysis ===")

    # Results analysis using the function
    results = analyze_league_results(df, league_name)
    league_results.append(results)
    print(f"Matches: {results['total_matches']:,} ({results['season_range']})")
    print(f"Home wins: {results['home_win_pct']:.1f}%, Draws: {results['draw_pct']:.1f}%, Away wins: {results['away_win_pct']:.1f}%")

    # Goals analysis
    goals = analyze_league_goals(df, league_name)
    league_goals.append(goals)
    print(f"Avg goals per match: {goals['avg_total_goals']:.2f} ± {goals['std_total_goals']:.2f}")
    print(f"Home: {goals['avg_home_goals']:.2f} ± {goals['std_home_goals']:.2f}, Away: {goals['avg_away_goals']:.2f} ± {goals['std_away_goals']:.2f}")

    # Odds analysis
    odds = analyze_league_odds(df, league_name)
    league_odds.append(odds)
    if odds['matches_with_odds'] > 0:
        print(f"Odds coverage: {odds['odds_coverage_pct']:.1f}% ({odds['matches_with_odds']:,} matches)")
        print(f"Avg odds - Home: {odds['avg_home_odds']:.2f}, Draw: {odds['avg_draw_odds']:.2f}, Away: {odds['avg_away_odds']:.2f}")
    else:
        print("No betting odds data available")

# Convert to DataFrames for easier manipulation
results_df = pd.DataFrame(league_results)
goals_df = pd.DataFrame(league_goals)
odds_df = pd.DataFrame(league_odds)

print("\n=== Individual League Analysis Complete ===")


=== Premier League Analysis ===
Matches: 9,410 (2000-2024)
Home wins: 45.8%, Draws: 24.6%, Away wins: 29.6%
Avg goals per match: 2.72 ± 1.67
Home: 1.53 ± 1.30, Away: 1.18 ± 1.16
Odds coverage: 99.1% (9,327 matches)
Avg odds - Home: 2.74, Draw: 3.97, Away: 4.73

=== Serie A Analysis ===
Matches: 9,012 (2000-2024)
Home wins: 44.6%, Draws: 27.1%, Away wins: 28.3%
Avg goals per match: 2.68 ± 1.65
Home: 1.50 ± 1.23, Away: 1.17 ± 1.11
Odds coverage: 98.7% (8,893 matches)
Avg odds - Home: 2.61, Draw: 3.69, Away: 4.56

=== Bundesliga Analysis ===
Matches: 7,522 (2000-2024)
Home wins: 45.7%, Draws: 24.7%, Away wins: 29.7%
Avg goals per match: 2.95 ± 1.72
Home: 1.67 ± 1.36, Away: 1.28 ± 1.20
Odds coverage: 99.0% (7,447 matches)
Avg odds - Home: 2.54, Draw: 3.92, Away: 4.31

=== La Liga Analysis ===
Matches: 9,008 (2000-2024)
Home wins: 47.1%, Draws: 25.2%, Away wins: 27.7%
Avg goals per match: 2.67 ± 1.68
Home: 1.55 ± 1.31, Away: 1.13 ± 1.11
Odds coverage: 99.0% (8,918 matches)
Avg odds - Home:

## Comparative Analysis

In [5]:
# Overall statistics
total_matches = results_df['total_matches'].sum()
overall_home_wins = results_df['home_wins'].sum()
overall_draws = results_df['draws'].sum()
overall_away_wins = results_df['away_wins'].sum()

overall_stats = {
    'total_matches': total_matches,
    'overall_home_win_pct': (overall_home_wins / total_matches) * 100,
    'overall_draw_pct': (overall_draws / total_matches) * 100,
    'overall_away_win_pct': (overall_away_wins / total_matches) * 100
}

print(f"=== Overall Statistics Across All Leagues ===")
print(f"Total matches: {overall_stats['total_matches']:,}")
print(f"Home win rate: {overall_stats['overall_home_win_pct']:.1f}%")
print(f"Draw rate: {overall_stats['overall_draw_pct']:.1f}%")
print(f"Away win rate: {overall_stats['overall_away_win_pct']:.1f}%")

# League rankings
print("\n=== League Rankings ===")
print("\nMost competitive (highest away win %):")
for i, row in results_df.sort_values('away_win_pct', ascending=False).iterrows():
    print(f"{row['league']}: {row['away_win_pct']:.1f}%")

print("\nHighest scoring (most goals per match):")
for i, row in goals_df.sort_values('avg_total_goals', ascending=False).iterrows():
    print(f"{row['league']}: {row['avg_total_goals']:.2f} goals/match")

print("\nMost draws:")
for i, row in results_df.sort_values('draw_pct', ascending=False).iterrows():
    print(f"{row['league']}: {row['draw_pct']:.1f}%")

=== Overall Statistics Across All Leagues ===
Total matches: 34,952
Home win rate: 45.8%
Draw rate: 25.4%
Away win rate: 28.8%

=== League Rankings ===

Most competitive (highest away win %):
Bundesliga: 29.7%
Premier League: 29.6%
Serie A: 28.3%
La Liga: 27.7%

Highest scoring (most goals per match):
Bundesliga: 2.95 goals/match
Premier League: 2.72 goals/match
Serie A: 2.68 goals/match
La Liga: 2.67 goals/match

Most draws:
Serie A: 27.1%
La Liga: 25.2%
Bundesliga: 24.7%
Premier League: 24.6%


## Generate Markdown Report

In [6]:
def generate_markdown_report(results_df, goals_df, odds_df, overall_stats, output_path):
    """Generate comprehensive markdown report."""
    
    # Calculate temporal coverage
    all_seasons = []
    for df in league_data.values():
        all_seasons.extend(df['season'].unique())
    min_season = min(all_seasons)
    max_season = max(all_seasons)
    
    # Calculate overall goal statistics (weighted by matches)
    total_goals = sum(df['total_goals'].sum() for df in league_data.values())
    overall_avg_goals = total_goals / total_matches
    
    # Calculate overall odds coverage
    total_matches_with_odds = odds_df['matches_with_odds'].sum()
    overall_odds_coverage = (total_matches_with_odds / total_matches) * 100
    
    report = f"""# European Soccer Data - Descriptive Statistics

*Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*

This report presents comprehensive descriptive statistics for European soccer match data across four major leagues:
- **Premier League** (England)
- **Serie A** (Italy) 
- **Bundesliga** (Germany)
- **La Liga** (Spain)

## Dataset Overview

The analysis covers **{total_matches:,} matches** across four major European leagues from {min_season}-{max_season}:
- **Temporal coverage**: {min_season}-{max_season} seasons
- **Betting odds coverage**: {overall_odds_coverage:.1f}% of matches
- **Geographic coverage**: England, Italy, Germany, Spain

### League Coverage
"""
    
    for _, row in results_df.iterrows():
        report += f"- **{row['league']}**: {row['total_matches']:,} matches ({row['season_range']})\n"
    
    report += f"""
## 1. Match Results Analysis

### Result Distribution by League

| League | Total Matches | Home Wins (%) | Draws (%) | Away Wins (%) |
|--------|---------------|---------------|-----------|---------------|
"""
    
    for _, row in results_df.iterrows():
        report += f"| {row['league']} | {row['total_matches']:,} | {row['home_win_pct']:.1f} | {row['draw_pct']:.1f} | {row['away_win_pct']:.1f} |\n"
    
    report += f"""
### Overall Statistics (All Leagues Combined)

- **Total Matches**: {overall_stats['total_matches']:,}
- **Home Win Rate**: {overall_stats['overall_home_win_pct']:.1f}%
- **Draw Rate**: {overall_stats['overall_draw_pct']:.1f}%
- **Away Win Rate**: {overall_stats['overall_away_win_pct']:.1f}%

### Key Findings - Match Results

- **Home advantage** is evident across all leagues (average {overall_stats['overall_home_win_pct']:.1f}% home win rate)
- **Most competitive league**: {results_df.loc[results_df['away_win_pct'].idxmax(), 'league']} ({results_df['away_win_pct'].max():.1f}% away wins)
- **Strongest home advantage**: {results_df.loc[results_df['home_win_pct'].idxmax(), 'league']} ({results_df['home_win_pct'].max():.1f}% home wins)
- **Most draws**: {results_df.loc[results_df['draw_pct'].idxmax(), 'league']} ({results_df['draw_pct'].max():.1f}% draws)

## 2. Goal Statistics Analysis

### Average Goals by League

| League | Avg Home Goals | Std Home Goals | Avg Away Goals | Std Away Goals | Avg Total Goals | Std Total Goals |
|--------|----------------|----------------|----------------|----------------|-----------------|------------------|
"""
    
    for _, row in goals_df.iterrows():
        report += f"| {row['league']} | {row['avg_home_goals']:.2f} | {row['std_home_goals']:.2f} | {row['avg_away_goals']:.2f} | {row['std_away_goals']:.2f} | {row['avg_total_goals']:.2f} | {row['std_total_goals']:.2f} |\n"
    
    # Calculate weighted averages for overall stats
    total_home_goals = sum(df['hometeamgoals'].sum() for df in league_data.values())
    total_away_goals = sum(df['awayteamgoals'].sum() for df in league_data.values())
    overall_avg_home = total_home_goals / total_matches
    overall_avg_away = total_away_goals / total_matches
    
    report += f"""
### Overall Goal Statistics

- **Average Home Goals per Match**: {overall_avg_home:.2f}
- **Average Away Goals per Match**: {overall_avg_away:.2f}
- **Average Total Goals per Match**: {overall_avg_goals:.2f}

### Key Findings - Goal Statistics

- **Highest scoring league**: {goals_df.loc[goals_df['avg_total_goals'].idxmax(), 'league']} ({goals_df['avg_total_goals'].max():.2f} goals/match)
- **Lowest scoring league**: {goals_df.loc[goals_df['avg_total_goals'].idxmin(), 'league']} ({goals_df['avg_total_goals'].min():.2f} goals/match)
- **Home teams score more**: {overall_avg_home:.2f} vs {overall_avg_away:.2f} goals per match
- **Goal range across leagues**: {goals_df['avg_total_goals'].min():.2f} - {goals_df['avg_total_goals'].max():.2f} goals/match

## 3. Betting Odds Analysis

### Average Betting Odds by League

| League | Matches with Odds | Coverage (%) | Avg Home Odds | Std Home Odds | Avg Draw Odds | Std Draw Odds | Avg Away Odds | Std Away Odds |
|--------|-------------------|--------------|---------------|---------------|---------------|---------------|---------------|---------------|
"""
    
    for _, row in odds_df.iterrows():
        if row['matches_with_odds'] > 0:
            report += f"| {row['league']} | {row['matches_with_odds']:,} | {row['odds_coverage_pct']:.1f} | {row['avg_home_odds']:.2f} | {row['std_home_odds']:.2f} | {row['avg_draw_odds']:.2f} | {row['std_draw_odds']:.2f} | {row['avg_away_odds']:.2f} | {row['std_away_odds']:.2f} |\n"
        else:
            report += f"| {row['league']} | 0 | 0.0 | - | - | - | - | - | - |\n"
    
    # Calculate overall odds averages (only for leagues with odds data)
    odds_leagues = odds_df[odds_df['matches_with_odds'] > 0]
    if len(odds_leagues) > 0:
        weighted_home_odds = sum(row['avg_home_odds'] * row['matches_with_odds'] for _, row in odds_leagues.iterrows()) / total_matches_with_odds
        weighted_draw_odds = sum(row['avg_draw_odds'] * row['matches_with_odds'] for _, row in odds_leagues.iterrows()) / total_matches_with_odds
        weighted_away_odds = sum(row['avg_away_odds'] * row['matches_with_odds'] for _, row in odds_leagues.iterrows()) / total_matches_with_odds
        
        report += f"""
### Betting Market Insights

- **Overall odds coverage**: {overall_odds_coverage:.1f}% ({total_matches_with_odds:,} matches)
- **Average Home Odds**: {weighted_home_odds:.2f}
- **Average Draw Odds**: {weighted_draw_odds:.2f}
- **Average Away Odds**: {weighted_away_odds:.2f}

### Key Findings - Betting Odds

- **Lowest home odds** (strongest home favorites): {odds_leagues.loc[odds_leagues['avg_home_odds'].idxmin(), 'league']} ({odds_leagues['avg_home_odds'].min():.2f})
- **Highest away odds** (weakest away teams): {odds_leagues.loc[odds_leagues['avg_away_odds'].idxmax(), 'league']} ({odds_leagues['avg_away_odds'].max():.2f})
- **Market expectation**: Home wins favored (lower odds), away wins least likely (higher odds)
"""
    
    report += f"""
## 4. League Comparative Rankings

### Competitive Balance (by Away Win %)
"""
    for i, (_, row) in enumerate(results_df.sort_values('away_win_pct', ascending=False).iterrows(), 1):
        report += f"{i}. **{row['league']}**: {row['away_win_pct']:.1f}% away wins\n"
    
    report += "\n### Goal Scoring (by Total Goals per Match)\n"
    for i, (_, row) in enumerate(goals_df.sort_values('avg_total_goals', ascending=False).iterrows(), 1):
        report += f"{i}. **{row['league']}**: {row['avg_total_goals']:.2f} goals/match\n"
    
    report += "\n### Home Advantage Strength (by Home Win %)\n"
    for i, (_, row) in enumerate(results_df.sort_values('home_win_pct', ascending=False).iterrows(), 1):
        report += f"{i}. **{row['league']}**: {row['home_win_pct']:.1f}% home wins\n"
    
    report += f"""
## 5. Statistical Summary

### Dataset Characteristics

- **Sample Size**: {total_matches:,} matches across 4 leagues
- **Temporal Coverage**: {max_season - min_season + 1} seasons ({min_season}-{max_season})
- **Data Completeness**: 100% match results, {overall_odds_coverage:.1f}% betting odds
- **Geographic Scope**: Top 4 European football leagues

### Key Statistical Insights

1. **Home Advantage Universal**: All leagues show significant home advantage ({overall_stats['overall_home_win_pct']:.1f}% vs {overall_stats['overall_away_win_pct']:.1f}%)
2. **League Variation**: Away win rates vary from {results_df['away_win_pct'].min():.1f}% to {results_df['away_win_pct'].max():.1f}%
3. **Goal Scoring Consistency**: Average goals range {goals_df['avg_total_goals'].min():.2f}-{goals_df['avg_total_goals'].max():.2f} across leagues
4. **Market Coverage**: Betting data available for {overall_odds_coverage:.1f}% of matches

### Methodology Notes

- **Data Source**: Football-Data.co.uk processed CSV files
- **Analysis Period**: {min_season}-{max_season} seasons
- **Statistical Measures**: Arithmetic means and standard deviations
- **Missing Data**: Excluded from calculations (primarily betting odds)
- **Result Classification**: Win/Draw/Loss from home team perspective

---

*Analysis generated from European soccer match data using descriptive statistics methodology. Results represent match-level outcomes across regular season fixtures in major European leagues.*
"""
    
    # Write to file
    with open(output_path, 'w') as f:
        f.write(report)
    
    return report

# Generate the report
output_file = output_dir / 'descriptive_statistics.md'
report_content = generate_markdown_report(results_df, goals_df, odds_df, overall_stats, output_file)

print(f"\n=== Report Generated ===")
print(f"Output saved to: {output_file}")
print(f"Report length: {len(report_content)} characters")
print("\n=== Analysis Complete ===")


=== Report Generated ===
Output saved to: ../../../src/descriptive_statistics.md
Report length: 5312 characters

=== Analysis Complete ===
